# Bước 00: Hotfix Căn Chỉnh Thời Tiết Causal (Re-align Weather Hotfix)
Dự án: Tốt nghiệp - Energy Forecasting - Nhóm thực hiện: The Outliers


In [2]:
import os
import gc
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from pathlib import Path

INPUT_PATH = Path("../../data/mlmart_base/v3_preprocessing.parquet")
OUTPUT_PATH = Path("../../data/mlmart_base/v3_preprocessing.parquet")

WEATHER_COLUMNS = (
    'weather_id', 'weather_type_id', 'weather_timestamp', 'weather_is_day',
    'shortwave_radiation', 'direct_normal_irradiance', 'diffuse_solar_radiation',
    'temperature_c', 'cloud_cover_total', 'cloud_cover_low', 'cloud_cover_mid',
    'cloud_cover_high', 'wind_speed', 'precipitation_mm', 'sunshine_duration',
    'weather_code', 'weather_type_is_day', 'weather_condition', 'weather_description'
)
print("Đã nạp thư viện thành công.")


Đã nạp thư viện thành công.


In [3]:
# ── 1. Đọc dữ liệu ──
df = pd.read_parquet(INPUT_PATH)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['weather_timestamp'] = pd.to_datetime(df['weather_timestamp'])

_delta_truoc = (df['weather_timestamp'] - df['timestamp']).dt.total_seconds() / 60
_leak_truoc = int((_delta_truoc > 0).sum())
print(f"Tổng số dòng: {len(df):,}")
print(f"Dòng dùng thời tiết TƯƠNG LAI trước khi sửa: {_leak_truoc:,}/{len(df):,} ({_leak_truoc / len(df) * 100:.2f}%)")


Tổng số dòng: 2,731,946
Dòng dùng thời tiết TƯƠNG LAI trước khi sửa: 0/2,731,946 (0.00%)


In [4]:
# ── Căn chỉnh lại thời tiết Causal (Code NGUYÊN BẢN từ srcs/00_utils/04_realign_mlmart_weather.py) ──
WEATHER_COLUMNS = (
    'weather_id', 'weather_type_id', 'weather_timestamp', 'weather_is_day',
    'shortwave_radiation', 'direct_normal_irradiance', 'diffuse_solar_radiation',
    'temperature_c', 'cloud_cover_total', 'cloud_cover_low', 'cloud_cover_mid',
    'cloud_cover_high', 'wind_speed', 'precipitation_mm', 'sunshine_duration',
    'weather_code', 'weather_type_is_day', 'weather_condition', 'weather_description'
)
LOOKUP_KEY = ("site_id", "_weather_hour")

# 1. Trích xuất bảng tra thời tiết chuẩn tại minute 00 (Hàm load_hourly_lookup)
_frame_m0 = df[pd.to_datetime(df['timestamp']).dt.minute.eq(0)].copy()
_frame_m0['_weather_hour'] = pd.to_datetime(_frame_m0['timestamp'], errors='raise')
_lookup = _frame_m0[[*LOOKUP_KEY, *[c for c in WEATHER_COLUMNS if c in df.columns]]].sort_values([*LOOKUP_KEY, 'weather_id'], kind='stable')
_lookup = _lookup.drop_duplicates(list(LOOKUP_KEY), keep='first').set_index(list(LOOKUP_KEY))

# 2. Realign thời tiết theo mốc timestamp.dt.floor('h') (Hàm realign_batch)
_timestamp = pd.to_datetime(df['timestamp'], errors='raise')
_keys = pd.MultiIndex.from_arrays(
    [df['site_id'].to_numpy(), _timestamp.dt.floor('h').to_numpy()],
    names=LOOKUP_KEY
)
_aligned = _lookup.reindex(_keys).reset_index(drop=True)

for col in WEATHER_COLUMNS:
    if col in df.columns:
        df[col] = _aligned[col].to_numpy()

df['weather_timestamp'] = _timestamp.dt.floor('h')

del _frame_m0, _lookup, _keys, _aligned
gc.collect()


0

In [5]:
# ── 3. Kiểm tra kết quả SAU khi sửa và Cổng kiểm soát (Assertion) ──
_delta_sau = (df['weather_timestamp'] - df['timestamp']).dt.total_seconds() / 60
_leak_sau = int((_delta_sau > 0).sum())
print(f"Dòng dùng thời tiết TƯƠNG LAI sau khi sửa: {_leak_sau:,} (Phải bằng 0)")

assert _leak_sau == 0, "LỖI BẢO VỆ: Vẫn còn dòng rò rỉ thời tiết tương lai!"
print("CỔNG KIỂM TRÁ: ĐẠT — 100% Causal Non-leaking Weather Data!")


Dòng dùng thời tiết TƯƠNG LAI sau khi sửa: 0 (Phải bằng 0)
CỔNG KIỂM TRÁ: ĐẠT — 100% Causal Non-leaking Weather Data!


In [6]:
# ── 4. Ghi đè file đĩa an toàn ──
df.to_parquet(OUTPUT_PATH, index=False)
print(f"HOÀN TẤT: Đã ghi đè file dữ liệu hotfix tại: {OUTPUT_PATH}")


HOÀN TẤT: Đã ghi đè file dữ liệu hotfix tại: ../../data/mlmart_base/v3_preprocessing.parquet
